# The master tables — every scored run, decodability BEFORE editability

Reads the `scores.json` files `master_eval.ipynb` writes (same scan: `runs/**`, excluding
`runs/archive/` and `_`-prefixed topics) and renders the cross-run tables, grouped by
topic subdirectory, with an environment column. Deliberately short: every future bespoke
table is a view/subset of these.

**Order is deliberate: decodability first.** An editability number is only interpretable
once the probes demonstrably read the state — an editor writing through a probe that
decodes nothing produces noise, not a negative result. So: **Table 1** decodability
(Probe Skill, the cross-environment axis: 1 = perfect, 0 = trivial baseline; ≡ R² on
regression), with the MLP ≥ linear tripwire count beside it; **Table 1b** discworld
decodability BY COMPONENT (position vs velocity per object — the aggregate is
variance-weighted ~1000:1 toward position, so it hides velocity; Othello's per-tile
equivalent is 64 columns and lives in each run's `scores.json` instead); **Table 2**
editability — each editor's best arm read against the run's unedited EI, never without
its guard (fidelity > 1 on discworld, or li-vs-pre collapsing on Othello, means the
"edit" degraded the model).

In [ ]:
# [1] Collect every scores.json (runs/**; archive/ and _topics excluded), flatten per run.
import json
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EDITORS = ("PI", "ND", "ND-sub", "GS", "GS-mine")

rows, perdim_rows, sources = [], [], {}
for sp in sorted((REPO / "runs").rglob("scores.json")):
    rel = sp.relative_to(REPO / "runs")
    if rel.parts[0] == "archive" or rel.parts[0].startswith("_"):
        continue
    s = json.loads(sp.read_text())
    row = {"topic": rel.parts[0], "run": sp.parent.name, "env": s["env"],
           "arch": s["arch"], "val": s["val_loss"], "eval": s["eval_version"]}
    if s["env"] == "discworld":
        T = s["targets"]["pos"]                      # headline target for the summary
        F = s["targets"].get("full")
        row |= {"skill_lin": max(T["probe_skill_linear"]),
                "skill_mlp": max(T["probe_skill_mlp"]),
                "skill_lin_seq": None, "skill_mlp_seq": None,   # DW is ALWAYS seq-split
                "tripwire": (T["probe_sanity"]["n_violations"]
                             + (F["probe_sanity"]["n_violations"] if F else 0)),
                "unedited_EI": T["unedited"]["edit_index"]}
        sources[sp.parent.name] = {"PI": "pos|linear (seq split)",
                                   "ND": "pos|linear (seq split)",
                                   "GS": "pos|mlp128 (seq split)"}
        if F:  # per-component decodability from the FULL target, at its best point
            bp = max(range(len(F["probe_skill_linear"])),
                     key=lambda i: F["probe_skill_linear"][i])
            for name, pd_ in (("LIN", F["probe_perdim_linear"][bp]),
                              ("MLP", F["probe_perdim_mlp"][bp])):
                perdim_rows.append({"run": sp.parent.name, "probe": name, "point": bp,
                                    **dict(zip(("o1·x", "o1·y", "o2·x", "o2·y",
                                                "o1·vx", "o1·vy", "o2·vx", "o2·vy"),
                                               pd_[:8]))})
        for ed in EDITORS:
            b = T["best"].get(ed)
            row[f"{ed}_EI"] = b["edit_index"] if b else None
            row[f"{ed}_arm"] = f"pt{b['point']} α{b['alpha']:g}" if b else None
            row[f"{ed}_guard"] = b["fidelity_ratio"] if b else None
    else:
        sk = s["probe_skill"]
        row |= {"skill_lin": max(sk.get("mine|linear|frame", [float('nan')])),
                "skill_mlp": max(sk.get("state|mlp|frame", [float('nan')])),
                "skill_lin_seq": max(sk.get("mine|linear|sequence", [float('nan')])),
                "skill_mlp_seq": max(sk.get("state|mlp|sequence", [float('nan')])),
                "tripwire": 0,
                "unedited_EI": s["unedited"]["edit_index_union"],
                "legal_mass": s["gates"]["legal_mass"],
                "ce_excess": s["gates"]["ce"] - s["gates"]["bayes_ce"]}
        sources[sp.parent.name] = s.get("probe_sources", {
            "PI": "mine|linear|frame", "ND": "mine|linear|frame",
            "GS": "state|mlp|frame"})
        for ed in EDITORS:
            b = s["best"].get(ed)
            row[f"{ed}_EI"] = b["edit_index_union"] if b else None
            row[f"{ed}_arm"] = f"pt{b['point']} α{b['alpha']:g}" if b else None
            row[f"{ed}_guard"] = b["li_error_vs_pre"] if b else None
    rows.append(row)

DF = pd.DataFrame(rows)
PERDIM = pd.DataFrame(perdim_rows)
print(f"{len(DF)} scored runs collected")

In [ ]:
# [2] TABLE 1 — DECODABILITY. Read before the editability table means anything.
#     Probe Skill at the best residual point (≡ R² on regression; 1 = perfect,
#     0 = trivial baseline). Discworld probes are always sequence-split; Othello shows
#     frame (Li's anchor) AND sequence (this repo's convention) side by side.
from IPython.display import display

t1 = (DF[["topic", "run", "env", "arch", "val", "skill_lin", "skill_mlp",
          "skill_lin_seq", "skill_mlp_seq", "tripwire"]]
      .rename(columns={"val": "val loss", "skill_lin": "LIN", "skill_mlp": "MLP-128",
                       "skill_lin_seq": "LIN (seq)", "skill_mlp_seq": "MLP (seq)",
                       "tripwire": "tripwire ⚠"})
      .sort_values(["topic", "env", "run"]).set_index(["topic", "run"]))
sty = (t1.style
       .format({"val loss": "{:.4f}", "LIN": "{:+.3f}", "MLP-128": "{:+.3f}",
                "LIN (seq)": "{:+.3f}", "MLP (seq)": "{:+.3f}"}, na_rep="—")
       .background_gradient(subset=["LIN", "MLP-128"], cmap="Greens",
                            vmin=0.0, vmax=1.0)
       .map(lambda v: "background-color:#c0392b;color:white"
            if isinstance(v, (int, float)) and v > 0 else "", subset=["tripwire ⚠"])
       .set_caption("Table 1 — Decodability (Probe Skill, best residual point). "
                    "Probes: DW LIN/MLP = pos target; Othello LIN = mine tiles, "
                    "MLP = state tiles. A red tripwire cell = MLP < linear somewhere: "
                    "that run's decodability AND editability are untrusted until refit."))
display(sty)

In [ ]:
# [3] TABLE 1b — DISCWORLD DECODABILITY BY COMPONENT (full target, best residual point).
#     The aggregate skill is variance-weighted ~1000:1 toward position; velocity is
#     only visible here. (Othello's 64-tile equivalent stays in each scores.json.)
if len(PERDIM):
    t1b = PERDIM.set_index(["run", "probe", "point"])
    sty = (t1b.style
           .format("{:+.3f}")
           .background_gradient(cmap="Greens", vmin=0.0, vmax=1.0, axis=None)
           .set_caption("Table 1b — Discworld decodability by component (Probe Skill "
                        "per dim; full target, sequence-split, at the linear probe's "
                        "best residual point). Columns: object · axis — positions "
                        "then velocities, sim units."))
    display(sty)
else:
    print("no discworld runs with per-component decodability yet")

In [ ]:
# [4] TABLE 2 — EDITABILITY. Only interpretable where Table 1 shows working probes.
#     Each editor's BEST arm (EI, the arm's point/α, and its guard). Guards:
#     discworld g = fidelity ratio (>1 destructive); Othello g = li-error-vs-pre
#     (LOW = the model forgot the pre-edit world — destroyed, not steered).
#     The provenance table underneath states EXACTLY which probes each editor
#     steered through — GS vs GS-mine is the open target-frame question.
cols = ["topic", "run", "env", "unedited_EI"]
for ed in EDITORS:
    cols += [f"{ed}_EI", f"{ed}_arm", f"{ed}_guard"]
t2 = (DF[cols].sort_values(["topic", "env", "run"]).set_index(["topic", "run"])
      .rename(columns={"unedited_EI": "unedited EI",
                       **{f"{ed}_EI": f"{ed} EI" for ed in EDITORS},
                       **{f"{ed}_arm": f"{ed} arm" for ed in EDITORS},
                       **{f"{ed}_guard": f"{ed} g" for ed in EDITORS}}))
ei_cols = ["unedited EI"] + [f"{ed} EI" for ed in EDITORS]
sty = (t2.style
       .format({c: "{:+.3f}" for c in ei_cols}
               | {f"{ed} g": "{:.2f}" for ed in EDITORS}, na_rep="—")
       .background_gradient(subset=[f"{ed} EI" for ed in EDITORS], cmap="RdYlGn",
                            vmin=-0.8, vmax=0.8)
       .set_caption("Table 2 — Editability (best arm per editor; EI scale effectively "
                    "+0.82…−0.80). Read each EI against that run's unedited EI, and "
                    "never without its guard."))
display(sty)

src = pd.DataFrame(sources).T
src.index.name = "run"
display(src.style.set_caption("Table 2 provenance — the probes each editor steers "
                              "through (target | family | probe split)"))

# ⛔ Nothing is written here. Each run's scores.json is the SINGLE SOURCE OF TRUTH;
# a cached flat copy beside the notebook would be a second one that silently goes
# stale the moment any run is rescored. Downstream views should re-run cell [1] —
# it is a sub-second scan — or read the scores.json files directly.
print(f"{len(DF)} runs · source of truth: runs/<topic>/<run>/scores.json")